In [ ]:
# =====================================================
# TEST CLASSIFICATION DATALOADER
# =====================================================

test_cls_ds = ClassificationDataset(
    TEST_PATH
)

test_cls_loader = DataLoader(
    test_cls_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print("Train:", len(train_cls_ds))
print("Test :", len(test_cls_ds))

In [ ]:
# ============================================================
# PART-1 : MobileNetV3 BASELINE
# ============================================================

import timm
import torch
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# BUILD MODEL
# ------------------------------------------------------------

baseline_model = timm.create_model(
    "mobilenetv3_large_100",
    pretrained=True,
    num_classes=4
).to(device)

# ------------------------------------------------------------
# LOSS
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    baseline_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ------------------------------------------------------------
# TRAIN CONFIG
# ------------------------------------------------------------

EPOCHS = 10

best_acc = 0

train_loss_history = []
train_acc_history = []

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

print("="*60)
print("Training MobileNetV3 Baseline")
print("="*60)

for epoch in range(EPOCHS):

    baseline_model.train()

    running_loss = 0

    correct = 0

    total = 0

    pbar = tqdm(train_cls_loader)

    for imgs, labels in pbar:

        imgs = imgs.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = baseline_model(imgs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        pbar.set_description(
            f"Epoch {epoch+1}/{EPOCHS}"
        )

    epoch_loss = running_loss / len(train_cls_loader)

    epoch_acc = correct / total

    train_loss_history.append(
        epoch_loss
    )

    train_acc_history.append(
        epoch_acc
    )

    print(
        f"Loss={epoch_loss:.4f} "
        f"Acc={epoch_acc:.4f}"
    )

    if epoch_acc > best_acc:

        best_acc = epoch_acc

        torch.save(
            baseline_model.state_dict(),
            "mobilenetv3_baseline.pth"
        )

print("\nBest Train Acc:", best_acc)

# ============================================================
# LOAD BEST
# ============================================================

baseline_model.load_state_dict(

    torch.load(
        "mobilenetv3_baseline.pth",
        map_location=device
    )

)

baseline_model.eval()

# ============================================================
# TEST
# ============================================================

y_true = []

y_pred = []

with torch.no_grad():

    for imgs, labels in tqdm(test_cls_loader):

        imgs = imgs.to(device)

        outputs = baseline_model(imgs)

        preds = outputs.argmax(1)

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            preds.cpu().numpy()
        )

# ============================================================
# METRICS
# ============================================================

acc = accuracy_score(
    y_true,
    y_pred
)

prec = precision_score(
    y_true,
    y_pred,
    average="weighted"
)

rec = recall_score(
    y_true,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print("\n")
print("="*60)
print("MobileNetV3 Baseline Results")
print("="*60)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report\n")

print(

    classification_report(

        y_true,

        y_pred,

        target_names=train_cls_ds.classes,

        digits=4

    )

)

# ============================================================
# SAVE RESULT
# ============================================================

ablation_results = []

ablation_results.append({

    "Configuration":

    "MobileNetV3",

    "Accuracy":

    round(acc*100,4),

    "Precision":

    round(prec*100,4),

    "Recall":

    round(rec*100,4),

    "F1":

    round(f1*100,4)

})

print("\nSaved to ablation_results")

In [ ]:
# ============================================================
# PART-2A : MobileNetV3 + ECA
# ============================================================

import copy

# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

eca_model = MobileNetV3_ECA().to(device)

classifier = nn.Linear(
    960,
    4
).to(device)

model = nn.Sequential(
    eca_model,
    classifier
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

EPOCHS = 10

best_acc = 0

print("="*60)
print("Training MobileNetV3 + ECA")
print("="*60)

for epoch in range(EPOCHS):

    model.train()

    correct = 0
    total = 0

    running_loss = 0

    pbar = tqdm(train_cls_loader)

    for imgs, labels in pbar:

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    acc_train = correct / total

    print(
        f"Epoch {epoch+1} "
        f"Acc={acc_train:.4f}"
    )

    if acc_train > best_acc:

        best_acc = acc_train

        torch.save(
            model.state_dict(),
            "mobilenetv3_eca.pth"
        )

print("Best:", best_acc)

In [ ]:
# ============================================================
# TEST
# ============================================================

model.load_state_dict(

    torch.load(
        "mobilenetv3_eca.pth",
        map_location=device
    )

)

model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for imgs, labels in tqdm(test_cls_loader):

        imgs = imgs.to(device)

        outputs = model(imgs)

        preds = outputs.argmax(1)

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            preds.cpu().numpy()
        )

acc = accuracy_score(
    y_true,
    y_pred
)

prec = precision_score(
    y_true,
    y_pred,
    average="weighted"
)

rec = recall_score(
    y_true,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print(acc, prec, rec, f1)

ablation_results.append({

    "Configuration":

    "MobileNetV3 + ECA",

    "Accuracy":

    round(acc*100,4),

    "Precision":

    round(prec*100,4),

    "Recall":

    round(rec*100,4),

    "F1":

    round(f1*100,4)

})

In [ ]:
# ============================================================
# LOAD SSL ENCODER
# ============================================================

ssl_encoder = MobileNetV3_ECA()

state = torch.load(
    "ssl_encoder.pth",
    map_location=device
)

new_state = {}

for k, v in state.items():

    if k.startswith("encoder."):

        new_state[
            k.replace(
                "encoder.",
                ""
            )
        ] = v

ssl_encoder.load_state_dict(
    new_state,
    strict=False
)

ssl_encoder = ssl_encoder.to(device)

print("SSL Encoder Loaded")

In [ ]:
for p in ssl_encoder.parameters():

    p.requires_grad = False

In [ ]:
linear_model = nn.Sequential(

    ssl_encoder,

    nn.Linear(
        960,
        4
    )

).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(

    linear_model[1].parameters(),

    lr=1e-3

)

In [ ]:
EPOCHS = 30

for epoch in range(EPOCHS):

    linear_model.train()

    correct = 0
    total = 0

    for imgs, labels in tqdm(train_cls_loader):

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = linear_model(imgs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    print(
        epoch+1,
        correct/total
    )

In [ ]:
linear_model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for imgs, labels in tqdm(test_cls_loader):

        imgs = imgs.to(device)

        outputs = linear_model(imgs)

        preds = outputs.argmax(1)

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            preds.cpu().numpy()
        )

acc = accuracy_score(
    y_true,
    y_pred
)

prec = precision_score(
    y_true,
    y_pred,
    average="weighted"
)

rec = recall_score(
    y_true,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print(acc, prec, rec, f1)

ablation_results.append({

    "Configuration":

    "MobileNetV3 + ECA + SSL",

    "Accuracy":

    round(acc*100,4),

    "Precision":

    round(prec*100,4),

    "Recall":

    round(rec*100,4),

    "F1":

    round(f1*100,4)

})

In [ ]:
# ============================================================
# PART-3 : Fine-Tuning + Proposed + Save Table + Plot
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# ------------------------------------------------------------
# LOAD FINETUNED ENCODER
# ------------------------------------------------------------

finetuned_encoder = MobileNetV3_ECA().to(device)

finetuned_encoder.load_state_dict(
    torch.load(
        "finetuned_encoder.pth",
        map_location=device
    )
)

print("Fine-Tuned Encoder Loaded")

# ------------------------------------------------------------
# BUILD CLASSIFIER
# ------------------------------------------------------------

finetune_eval_model = nn.Sequential(

    finetuned_encoder,

    nn.Linear(
        960,
        4
    )

).to(device)

# ------------------------------------------------------------
# QUICK TRAIN ONLY FINAL LAYER
# (Linear Evaluation on Fine-Tuned Encoder)
# ------------------------------------------------------------

for p in finetune_eval_model[0].parameters():
    p.requires_grad = False

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    finetune_eval_model[1].parameters(),
    lr=1e-3
)

EPOCHS = 5

print("="*60)
print("Training Linear Head on Fine-Tuned Encoder")
print("="*60)

for epoch in range(EPOCHS):

    finetune_eval_model.train()

    correct = 0
    total = 0

    for imgs, labels in train_cls_loader:

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = finetune_eval_model(imgs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

    print(
        f"Epoch {epoch+1}: {correct/total:.4f}"
    )

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

finetune_eval_model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for imgs, labels in test_cls_loader:

        imgs = imgs.to(device)

        outputs = finetune_eval_model(imgs)

        preds = outputs.argmax(1)

        y_true.extend(
            labels.numpy()
        )

        y_pred.extend(
            preds.cpu().numpy()
        )

acc = accuracy_score(
    y_true,
    y_pred
)

prec = precision_score(
    y_true,
    y_pred,
    average="weighted"
)

rec = recall_score(
    y_true,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

print("\n")
print("="*60)
print("MobileNetV3 + ECA + Fine-Tuning")
print("="*60)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report\n")

print(

    classification_report(

        y_true,

        y_pred,

        target_names=train_cls_ds.classes,

        digits=4

    )

)

# ------------------------------------------------------------
# SAVE RESULT
# ------------------------------------------------------------

ablation_results.append({

    "Configuration":

    "MobileNetV3 + ECA + Fine-Tuning",

    "Accuracy":

    round(acc*100,4),

    "Precision":

    round(prec*100,4),

    "Recall":

    round(rec*100,4),

    "F1":

    round(f1*100,4)

})

# ------------------------------------------------------------
# ADD PROPOSED MODEL RESULT
# (Use your ProtoNet result here)
# ------------------------------------------------------------

ablation_results.append({

    "Configuration":

    "Proposed (SSL + Fine-Tuning + ProtoNet)",

    "Accuracy":99.6806,

    "Precision":99.6818,

    "Recall":99.6806,

    "F1":99.6807

})

# ------------------------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame(
    ablation_results
)

print("\n")
print("="*70)
print("ABLATION STUDY")
print("="*70)

print(df)

# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

df.to_csv(
    "ablation_study.csv",
    index=False
)

print("\nSaved -> ablation_study.csv")

# ------------------------------------------------------------
# SAVE EXCEL
# ------------------------------------------------------------

df.to_excel(
    "ablation_study.xlsx",
    index=False
)

print("Saved -> ablation_study.xlsx")

# ------------------------------------------------------------
# BAR PLOT
# ------------------------------------------------------------

plt.figure(figsize=(10,6))

plt.bar(
    df["Configuration"],
    df["Accuracy"]
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.ylabel("Accuracy (%)")

plt.title("Ablation Study")

plt.grid(axis="y")

plt.tight_layout()

plt.show()

# ------------------------------------------------------------
# PAPER READY TABLE
# ------------------------------------------------------------

print("\n")
print(df.to_markdown(index=False))

print("\n")
print("="*70)
print("Ablation Study Complete")
print("="*70)